# Earthquake Vision — Malatya hasar segmentasyonu: Colab eğitimi

Bu not defteri sırasıyla şunları yapar:

1. GPU kontrolü
2. GitHub reposunu klonlama (private repo için token gerekir)
3. **Veri denetimi** (`audit_dataset.py`): KATE-CD içinde Malatya'yı kanıtlar. `PASSED` olmazsa devam edilmez.
4. Veri hazırlama + YOLO formatına dönüştürme
5. İki deney:
   - **A — yalnızca Malatya:** train 53, val 5, test 4
   - **B — 7 şehirle eğit, yalnızca Malatya'da ölç:** train 404, val 5 (Malatya), test 4 (Malatya)
6. Değerlendirme (mAP conf=0.001, piksel IoU/Dice + %95 güven aralığı) ve örnek paneller
7. Ağırlıkları ve raporları indirme

**Başlamadan önce:** Menüden *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seçin.

## 1. GPU kontrolü

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU YOK: Çalışma zamanı türünü T4 GPU yapın"

## 2. Repoyu klonla

Repo private olduğu için bir GitHub token'ı gerekir: GitHub → *Settings → Developer settings → Personal access tokens → Fine-grained tokens* → yalnızca bu repo için **Contents: Read** izni. Token aşağıda gizli olarak sorulur ve not defterine kaydedilmez.

In [ ]:
import getpass, os, subprocess
REPO = "selimhancil/earthquake-damage-segmentation"
if not os.path.isdir("earthquake-damage-segmentation"):
    token = getpass.getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git"], check=True)
    del token
%cd earthquake-damage-segmentation
!git log --oneline -1

## 3. Paketler
Colab'da PyTorch hazır gelir; yalnızca Ultralytics kurulur (requirements.txt Mac için sabitlenmiştir).

In [ ]:
!pip install -q ultralytics
import ultralytics, torch, cv2
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

## 4. Veri denetimi (zorunlu kapı)

In [ ]:
!python -m src.training.audit_dataset
import json
audit = json.load(open("reports/data_audit.json"))
assert audit["status"] == "PASSED", "Denetim geçmedi; eğitim yapılmayacak: " + str(audit["blockers"])
print("Malatya:", audit["malatya"]["by_split"])

## 5. Veri hazırlama ve YOLO dönüştürme (iki varyant)

In [ ]:
!python -m src.training.prepare_malatya --variant malatya
!python -m src.training.convert_to_yolo --variant malatya
!python -m src.training.prepare_malatya --variant kate7_eval_malatya
!python -m src.training.convert_to_yolo --variant kate7_eval_malatya

## 6A. Deney A — yalnızca Malatya ile eğitim

Şartnamedeki başlangıç ayarları: `imgsz=512, epochs=30, batch=8, patience=10, seed=42`. Bunlar başlangıç değerleridir, performans garantisi değildir.

In [ ]:
!python -m src.training.train --variant malatya --name A_malatya_only --device 0

## 6B. Deney B — 7 şehirle eğitim, Malatya'da değerlendirme

In [ ]:
!python -m src.training.train --variant kate7_eval_malatya --name B_kate7_eval_malatya --device 0

## 7. Değerlendirme (yalnızca Malatya test kümesi)

- mAP: `conf=0.001` (PR eğrisi için). Arayüzdeki 0.70 eşiği burada kullanılmaz.
- Piksel IoU/Dice: `conf ≥ 0.25 / 0.50 / 0.70` için ayrı ayrı, bootstrap %95 güven aralığıyla.
- Test kümesi 4 görüntü olduğu için sonuçları kesin değil, belirsizlik aralığıyla yorumlayın.

In [ ]:
!python -m src.training.evaluate --weights weights/A_malatya_only_best.pt --variant malatya --name A_malatya_only --device 0
!python -m src.training.evaluate --weights weights/B_kate7_eval_malatya_best.pt --variant kate7_eval_malatya --name B_kate7_eval_malatya --device 0

## 8. Karşılaştırma tablosu

In [ ]:
import json
rows = []
for name in ["A_malatya_only", "B_kate7_eval_malatya"]:
    r = json.load(open(f"reports/evaluation_{name}.json"))
    s = r.get("ultralytics_seg", {})
    m = r["pixel_metrics_by_conf_threshold"]["0.50"]
    rows.append((name, r["n_images"], s.get("mask_map50"), s.get("mask_map50_95"),
                 m["micro_iou"], m["micro_iou_ci95"], m["micro_dice_f1"], m["area"]["relative_area_error"]))
print(f"{'deney':24} {'n':>2} {'mAP50':>7} {'mAP50-95':>9} {'IoU@0.5':>8} {'IoU %95 GA':>16} {'Dice@0.5':>9} {'alan hatası':>11}")
for r in rows:
    print(f"{r[0]:24} {r[1]:>2} {str(r[2]):>7} {str(r[3]):>9} {str(r[4]):>8} {str(r[5]):>16} {str(r[6]):>9} {str(r[7]):>11}")

## 9. Örnek paneller (öncesi | sonrası | ground truth | tahmin)

In [ ]:
from IPython.display import Image, display
import glob
for path in sorted(glob.glob("reports/figures/*/*.png"))[:8]:
    print(path); display(Image(path))

## 10. Sonuçları indir

Uygulamada kullanacağınız modeli seçin ve `weights/best_model.pt` olarak bilgisayarınızdaki projeye koyun. Raporlar (`reports/`) makale için kanıt niteliğindedir; repoya da eklenebilir.

In [ ]:
!zip -q -r /content/earthquake_vision_results.zip weights/*_best.pt reports/ runs/segment/*/results.csv runs/segment/*/args.yaml
from google.colab import files
files.download("/content/earthquake_vision_results.zip")